In [ ]:
from matplotlib import font_manager
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import os, re, json, yaml, math, hashlib, collections, datetime

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)

NB_DIR = "/root/learning-notebooks/第15周"
SM_YAML = "/root/learning-notebooks/semantic-model/mi-cre-semantic-model-v0.1.yaml"
ONT_PATH = "/root/docs/lanlnk/config/ontology/business-ontology.yaml"
PACK = "/root/LnkChatBI/backend/scripts/mall_ops_starter_pack"
PG_SCHEMA = "/root/LnkChatBI/mallcre_pg_init/mallcre_postgres.sql"
DEMO_SCHEMA = "/root/LnkChatBI/postgres_demo_schema.sql"
SEED = "/root/LnkChatBI/mallcre_pg_init/mallcre_seed_realistic.sql"
OUT_DIR = "/root/learning-notebooks/semantic-model/consumers/lnkchatbi"
os.makedirs(OUT_DIR, exist_ok=True)
print("产物目录:", OUT_DIR)

# ⚡ W15-D3 · 实验3：Semantic Model v0.1.1 → LnkChatBI term-aliases / SQL 示例生成与校准

> 开发期 · Week 15「LnkChatBI 精读 × Semantic Model 第一个消费方」Day 3（2026-09-09 周三 · 动手实验日）
> 前置（D7 护栏）：① v0.1.1 三件套小修已原位 patch 并机器复验；② 本 notebook §1 先测导入前基线，再生成、再验证。
> Today's Question：**同一份 ontology，喂"术语库"和喂"表结构注释"效果差在哪？**（D2 已给一半答案：差的不是内容是通道能力——本实验用数字补另一半）

**实验链路**：v0.1.1 机器检查 → 导入前基线（starter 11 组 × 14 探针）→ 术语层收割（883+ 口径复算）→ 14 组绑定生成（词-物归并）→ 11 条 SQL 示例 → 对照 mallcre 模式/种子验证 → 生成后召回对比 → 落盘产物 + 指纹 manifest。


## 0. 前置：v0.1.1 三件套机器复验（R2 冻结 / R3 别名表 / G-01 模板 / 指纹锚）

W14-D7 整改清单挂账两天，今晨已原位 patch（meta.version=0.1.1）。生成器开工前先证明护栏在位：场景层冻结可机读、Context 别名表全解析、frontmatter 提案模板四键齐全、ontology 指纹未漂移（patch 不触碰 SoT）。

In [ ]:
SM = yaml.safe_load(open(SM_YAML))
assert SM["meta"]["version"] == "0.1.1", "v0.1.1 未落地"
assert SM["capability_policy"]["scenario_layer_frozen"] is True            # R2 场景层冻结
alias_sm = SM["entity"]["context_alias_v1"]
canon = set(SM["entity"]["contexts"])
assert alias_sm and all(v in canon for v in alias_sm.values())            # R3 别名表全解析
fm = SM["governance_proposals"]["ontology_frontmatter"]                    # G-01 提案模板
assert {"version", "status", "maintainer", "change_process"} <= set(fm)
sha_ont = hashlib.sha256(open(ONT_PATH, "rb").read()).hexdigest()[:16]
assert sha_ont == SM["sources"]["ontology"]["sha256_16"]                   # 指纹锚
print(f"v0.1.1 前置检查通过：R2 冻结 | R3 别名表 {len(alias_sm)} 条全解析 | G-01 frontmatter 四键 | ontology 指纹 {sha_ont} 未漂移")

## 1. 导入前基线（D7 护栏：没有基线的 Demo 是展示不是验证）

检索机制忠实模拟 D2 读出的真实行为：**单向子串**（问句包含词/别名才算命中，terminology.py:913 语义）+ **字符 n-gram 余弦模拟向量路**（阈值 0.4，pgvector 双路对照）。探针 = starter pack 自己的 10 条 P0 验收问句（BI 侧）+ 4 条 ERP 侧探针（含 A101 身份路径）。

In [ ]:
starter_terms = json.load(open(f"{PACK}/terminology.json"))
accept_qs = json.load(open(f"{PACK}/acceptance_questions.json"))
assert all(set(t) == {"word", "other_words", "description"} for t in starter_terms)

probe = [("BI-P0", q["question"]) for q in accept_qs] + [
    ("ERP-对象", "A101 铺位为什么不能出租？"),
    ("ERP-对象", "L1-01 临街铺现在的状态是什么？"),
    ("ERP-对象", "合同 CONT_DEMO_001 名下有哪些铺位？"),
    ("ERP-对象", "哪些铺位目前空置、面积多大？"),
]

def substring_recall(question, groups):
    """单向子串：问句包含 词/任一别名（LnkChatBI FILTER_TERMS 语义，命中任一拉全组）"""
    hit = []
    for g in groups:
        vocab = [g["word"]] + list(g.get("other_words") or [])
        if any(v and v in question for v in vocab):
            hit.append(g["word"])
    return hit

def ngram(s, ns=(2, 3, 4)):
    t = "".join(s.split()); c = collections.Counter()
    for n in ns:
        for i in range(len(t) - n + 1):
            c[t[i:i+n]] += 1
    return c

def cosine(a, b):
    common = set(a) & set(b)
    if not common: return 0.0
    na = math.sqrt(sum(v*v for v in a.values())); nb = math.sqrt(sum(v*v for v in b.values()))
    return sum(a[k]*b[k] for k in common) / (na * nb)

def vector_recall(question, groups, th=0.4):
    """模拟 pgvector 双路：问句向量 vs 组内词向量最佳相似度过阈值"""
    qv = ngram(question); out = []
    for g in groups:
        vocab = [g["word"]] + list(g.get("other_words") or [])
        best = max((cosine(qv, ngram(w)) for w in vocab), default=0.0)
        if best >= th: out.append(g["word"])
    return out

def recall_stats(groups):
    rows = []
    for cat, q in probe:
        rows.append(dict(cat=cat, q=q, substr=substring_recall(q, groups), vec=vector_recall(q, groups)))
    summary = collections.defaultdict(lambda: [0, 0])
    for r in rows:
        summary[r["cat"]][1] += 1
        if r["substr"] or r["vec"]: summary[r["cat"]][0] += 1
    return rows, dict(summary)

base_rows, base_sum = recall_stats(starter_terms)
print("=== 导入前基线：starter 11 组 × 14 探针问句 ===")
for r in base_rows:
    print(f"[{r['cat']}] {r['q'][:26]:<28} 子串={r['substr'] or '—'} 向量={r['vec'] or '—'}")
print("按类命中率:", {k: f"{a}/{b}" for k, (a, b) in base_sum.items()})

## 2. 术语层收割：从 ontology 到生成源（含口径复算）

收割规则（R2 冻结在位）：只读 `modules[].aliases`（模块级别名）与 `sub_functions[].terms`（子功能术语），**不读 scenarios**——15 条场景是半成品，只允许用于"排除校验"。同时复算术语层规模，对账 W14-D3 健康报告口径。

In [ ]:
ont = yaml.safe_load(open(ONT_PATH))
occ_alias = occ_sf = 0
loc = collections.defaultdict(set)          # term -> {(module, sub_function | <module-alias>)}
for m, spec in ont["modules"].items():
    for a in spec.get("aliases", []):
        loc[a].add((m, "<module-alias>")); occ_alias += 1
    for sf, ss in spec.get("sub_functions", {}).items():
        for t in ss.get("terms", []):
            loc[t].add((m, sf)); occ_sf += 1

total_occ = occ_alias + occ_sf
unique_terms = set(loc)
cross_mod = {t: v for t, v in loc.items() if len({m for m, _ in v}) > 1}
print(f"复算口径：术语出现 {total_occ}（sf 条目 {occ_sf} + 模块级 alias {occ_alias}）| 唯一词 {len(unique_terms)} | 跨模块复用 {len(cross_mod)}")
print(f"健康报告口径（W14-D3）：883 / 792 / 91 → 差异 = 模块级 aliases（{occ_alias} 出现、{len(unique_terms)-792} 唯一）未入报告计数 —— 治理头必须声明计数口径（G-01 教训+1）")
print("跨模块复用 Top5:", sorted(((t, len(v)) for t, v in cross_mod.items()), key=lambda x: -x[1])[:5])

scenario_names = [sc["name"] for spec in ont["modules"].values()
                  for sf in spec.get("sub_functions", {}).values()
                  for sc in sf.get("scenarios", [])]
print(f"scenario_entries = {len(scenario_names)}（R2 冻结：仅用于排除校验，禁止入生成源）")

## 3. 生成：14 组绑定规格（tier-1 人工锚定）+ 术语层自动归并（tier-2）

按 D2 定稿规格生成：**父行 = 规范词 + description（定义+口径+映射）**；**子行 = 别名**（ontology 术语自动归并 + 人工含编码风格值 A101→LOC_DEMO_L101——D2 实验证明这类映射两路检索都救不了，唯一出路是预喂）。
护栏：① XML 特殊字符全角化（to_xml_string 反转义面）；② 词长 ≤8 / 别名 ≤12（单向子串检索可行性）；③ 与 starter 组冲突的别名不硬塞，转为 merge-suggestion（组是归一语义单元边界，写错整组污染）；④ 跨模块歧义词（如「项目」×16）不回避，description 声明 demo 消歧口径（G-04 的消费侧示范）。

In [ ]:
TIER1 = [
 dict(word="铺位",
      definition="商场最小可租赁经营单元（物理锚 bi_d_position，一行一铺位；bilocation 为源系统镜像）。",
      caliber="状态口径 bi_d_position.POSITION_STATE 取值 已出租/空置；空置聚合 BI 侧走 vw_mall_ops_vacancy_snapshot（business_status=vacant，双口径并存）。",
      map_note="编码风格映射：口语 A101 型编码 → POSITION_CODE 前缀 LOC_DEMO_L1xx（demo 种子）；生产环境由铺位编码规则注册表替换。",
      bindings=["bi_d_position.POSITION_CODE","bi_d_position.POSITION_NAME","bi_d_position.POSITION_STATE","bilocation.CODE"],
      extra_aliases=["A101","L1-01","L1-02","L2-01","L2-02","LOC_DEMO_L101","LOC_DEMO_L102","LOC_DEMO_L201","LOC_DEMO_L202","门店铺位"]),
 dict(word="楼层",
      definition="楼宇内的经营分层，锚 bi_d_position.FLOOR_CODE/FLOOR_NAME（F1/L1 首层等），BI 侧维表 dim_floor。",
      caliber="demo 种子楼层编码 F1、F2；ERP 侧 FLOOR_CODE 与 BI 侧 floor_code 大小写折叠见导入说明。",
      bindings=["bi_d_position.FLOOR_CODE","bi_d_position.FLOOR_NAME","dim_floor.floor_name"], extra_aliases=["F1","F2"]),
 dict(word="楼宇",
      definition="项目下的物理楼栋，锚 bi_d_position.BUILDING_CODE/BUILDING_NAME。",
      caliber="demo 种子 BLDG_A（A座）；楼宇编码在项目内唯一。",
      bindings=["bi_d_position.BUILDING_CODE","bi_d_position.BUILDING_NAME"], extra_aliases=["BLDG_A","楼栋"]),
 dict(word="项目",
      definition="购物中心经营主体，demo 锚 bi_d_position.STORE_CODE/STORE_NAME 与 BI 侧 dim_project。",
      caliber="歧义声明（G-04）：「项目」在 ontology 跨 5 模块复用 16 处；本组 demo 口径=购物中心项目（MALL_DEMO_001 星河购物中心），集团→区域→项目组织层级属系统管理域不在本表。",
      bindings=["bi_d_position.STORE_CODE","bi_d_position.STORE_NAME","dim_project.project_name"],
      extra_aliases=["MALL_DEMO_001","星河购物中心","购物中心"], ambiguous=True),
 dict(word="商户",
      definition="与项目签约的经营主体（租户），ERP 锚 bi_b_tenant（TENANT_CODE/TENANT_NAME），BI 侧日粒度 vw_mall_ops_merchant_daily.merchant_name。",
      caliber="商户/租户在 ontology 两词并存，demo 语义等价，统一归并本组；租户账户另见 lease 侧。",
      bindings=["bi_b_tenant.TENANT_CODE","bi_b_tenant.TENANT_NAME","bi_b_tenant.SHORT_NAME"],
      extra_aliases=["租户","租户名称","商户名称"]),
 dict(word="品牌",
      definition="商户经营的商业品牌；门店侧招牌名锚 bi_d_position.SIGNBOARD，BI 侧品牌维 dim_brand.brand_name。",
      caliber="招牌名=门店级品牌展示名（SIGNBOARD），与品牌维（dim_brand）粒度不同，聚合统计用品牌维。",
      bindings=["bi_d_position.SIGNBOARD","dim_brand.brand_name"], extra_aliases=["招牌","招牌名"]),
 dict(word="租赁合同",
      definition="铺位租赁契约，ERP 锚 bi_d_contract.CONT_NO（合同维），铺位关联 bi_d_position.CONT_NO，BI 侧事实 fact_leasing_contract.contract_no。",
      caliber="合同身份口径 CONT_NO 前缀 CONT_；到期窗口查询走 starter 组「合同到期」（vw_mall_ops_contract_expiry.days_to_expiry），两组互补不重复。",
      bindings=["bi_d_contract.CONT_NO","bi_d_position.CONT_NO","fact_leasing_contract.contract_no"],
      extra_aliases=["合同","合同号","租约","合约","CONT_DEMO_001"]),
 dict(word="合同状态",
      definition="租赁合同生命周期状态。两套体系并存：ERP 侧 bi_d_contract.CONT_STATE，BI 侧 fact_leasing_contract.contract_status。",
      caliber="查询时以所在表为准——状态值域未对齐（D1 结论 L2 落点），跨表比较前先做状态映射。",
      bindings=["bi_d_contract.CONT_STATE","fact_leasing_contract.contract_status"], extra_aliases=["租约状态"]),
 dict(word="空置",
      definition="铺位未出租状态。ERP 口径 bi_d_position.POSITION_STATE=空置；BI 口径 fact_shop_daily_operation.business_status=vacant（视图 vw_mall_ops_vacancy_snapshot）。",
      caliber="双口径并存：铺位实时状态用 ERP 侧；空置面积快照/趋势用 BI 侧（gla_area 汇总）。",
      bindings=["bi_d_position.POSITION_STATE","vw_mall_ops_vacancy_snapshot.vacant_area"],
      extra_aliases=["空铺","空铺面积","空置铺位","未出租"]),
 dict(word="停车",
      definition="停车场经营数据，BI 锚 fact_parking_daily（车流/周转/收入），项目车位规模 dim_project.parking_spaces。",
      caliber="停车收入口径 fact_parking_daily.parking_revenue_amount（日粒度）；车流=parking_entries。",
      bindings=["fact_parking_daily.parking_entries","fact_parking_daily.parking_revenue_amount","dim_project.parking_spaces"],
      extra_aliases=["车位","停车场","停车收入","车流"]),
 dict(word="费用科目",
      definition="计费科目字典。ERP 锚 bisubject（STOREID 级，SUBJECTCODE/SUBJECTNAME/SUBJECT_TYPE），BI 侧维表 dim_fee_subject。",
      caliber="账单/押金明细的 SUBJECTCODE 均引用本字典；两套编码未对齐前以各自表内自洽为准。",
      bindings=["bisubject.SUBJECTCODE","bisubject.SUBJECTNAME","bisubject.SUBJECT_TYPE","dim_fee_subject.subject_name"],
      extra_aliases=["科目","收费科目","费项"]),
 dict(word="销售明细",
      definition="POS/销售流水明细，ERP 锚 bisaledtl（笔级，SALEYEAR/SALEMONTH/SALEDAY）与 bipossaledtl（POS 通道流水，无铺位列，按 CONTRACT_CODE/BUSDATE 关联合同）。",
      caliber="明细=笔级；汇总口径走 BI 侧 vw_mall_ops_merchant_daily（starter 组销售额），勿用明细表直算全场汇总（口径差异）；POS 通道表无 POSITIONCODE，铺位维度须经合同桥接。",
      bindings=["bisaledtl.POSITIONCODE","bisaledtl.SALEYEAR","bisaledtl.SALEMONTH","bipossaledtl.CONTRACT_CODE"],
      extra_aliases=["销售流水","POS 销售"]),
 dict(word="押金",
      definition="合同保证金类款项。ERP 流水锚 bipreddeposit（DEPOSITDATE/DSPTYPE，笔级），BI 侧合同字段 fact_leasing_contract.deposit_amount（合同级）。",
      caliber="两粒度勿混：流水=实收轨迹（笔级），合同字段=应缴额度（合同级）。",
      bindings=["bipreddeposit.DEPOSITDATE","bipreddeposit.DSPTYPE","fact_leasing_contract.deposit_amount"],
      extra_aliases=["保证金","押金流水"]),
 dict(word="账单",
      definition="应收账单头，ERP 锚 bibillrecvinfo（BILL_NUM/BILL_DATE/LASTPAYDATE，账单粒度）。",
      caliber="账单身份与日期归本组；应收/实收/欠费金额口径归 starter 组（vw_mall_ops_arrears_current），两组互补。",
      bindings=["bibillrecvinfo.BILL_NUM","bibillrecvinfo.BILL_DATE","bibillrecvinfo.LASTPAYDATE"],
      extra_aliases=["账单号","缴款日"]),
]

def xml_safe(s):
    return s.replace("<", "＜").replace(">", "＞").replace("&", "＆")

starter_words = {t["word"] for t in starter_terms}
blocked_vocab = [w for t in starter_terms for w in [t["word"]] + list(t["other_words"])]

generated, merge_suggestions = [], []
for spec in TIER1:
    word = spec["word"]
    absorbed = sorted(t for t in unique_terms if word in t and len(t) <= 8 and t != word)  # tier-2 自动归并
    other, seen = [], set()
    for a in absorbed + spec.get("extra_aliases", []):
        if a == word or a in seen or len(a) > 12:
            continue
        clash = [b for b in blocked_vocab if b in a]
        if clash:                                   # 与 starter 组语义冲突 → 转合并建议，不硬塞
            merge_suggestions.append(dict(alias=a, suggest_into=clash[0])); continue
        seen.add(a); other.append(a)
    desc = xml_safe(spec["definition"] + " 口径：" + spec["caliber"] + (" 映射：" + spec["map_note"] if spec.get("map_note") else ""))
    generated.append(dict(word=word, other_words=other, description=desc,
                          bindings=spec["bindings"], ambiguous=spec.get("ambiguous", False)))

consumed = {t for t in unique_terms if any(s["word"] in t for s in TIER1)}

words = [g["word"] for g in generated]
assert len(words) == len(set(words)) and not (set(words) & set(starter_words))
assert all(len(w) <= 8 for w in words)                                   # 单向子串可行性
assert not any(re.search(r"[<>&]", g["description"]) for g in generated) # XML 反转义面
assert not (set(words) & set(scenario_names))                            # R2 排除校验
print(f"生成 {len(generated)} 组 | other_words 共 {sum(len(g['other_words']) for g in generated)} 条 | merge-suggestions {len(merge_suggestions)} 条")
for g in generated:
    print(f"  {g['word']}{'(歧义声明)' if g['ambiguous'] else ''}: +{len(g['other_words'])} 别名")
print(f"术语层唯一词被消费（归并入组）: {len(consumed)}/{len(unique_terms)} = {len(consumed)/len(unique_terms):.1%}")
print("冲突转合并建议示例:", merge_suggestions[:3])

## 4. SQL 示例校准集生成（11 条，绑定已核实对象）

每条 = 问句 + SQL。问句刻意包含术语组的词/别名（检索与校准同源）；SQL 只引用上面核实过的物理列。A101 是身份路径（L1）样例，ERP/BI 双侧各有代表。

In [ ]:
SQL_EXAMPLES = [
 dict(question="A101 铺位为什么不能出租？",
      description="SELECT STORE_CODE, POSITION_CODE, POSITION_NAME, POSITION_STATE, CONT_NO, END_DATE FROM bi_d_position WHERE POSITION_CODE = 'LOC_DEMO_L101';"),
 dict(question="当前所有空置铺位有哪些？面积多大？",
      description="SELECT FLOOR_NAME, POSITION_CODE, POSITION_NAME, POSITION_TYPE, RENT_AREA FROM bi_d_position WHERE POSITION_STATE = '空置' ORDER BY RENT_AREA DESC;"),
 dict(question="合同 CONT_DEMO_001 名下有哪些铺位？",
      description="SELECT POSITION_CODE, POSITION_NAME, POSITION_STATE, CONT_NO, END_DATE FROM bi_d_position WHERE CONT_NO = 'CONT_DEMO_001';"),
 dict(question="L1 层各铺位的租金单价是多少？",
      description="SELECT POSITION_CODE, POSITION_NAME, RENT_PRICE, FIXFEE_PRICE FROM bi_d_position WHERE FLOOR_CODE = 'F1' ORDER BY RENT_PRICE DESC;"),
 dict(question="星河购物中心有哪些商户？",
      description="SELECT TENANT_CODE, TENANT_NAME, SHORT_NAME FROM bi_b_tenant WHERE STORE_ID = 'STORE_DEMO_001' ORDER BY TENANT_CODE;"),
 dict(question="费用科目有哪些？",
      description="SELECT SUBJECTCODE, SUBJECTNAME, SUBJECT_TYPE FROM bisubject WHERE STOREID = 'STORE_DEMO_001';"),
 dict(question="商户押金缴纳情况如何？",
      description="SELECT TENANTCODE, TENANTNAME, CONTRACTCODE, SUBJECTNAME, DEPOSITDATE, DSPTYPE FROM bipreddeposit WHERE STOREID = 'STORE_DEMO_001' ORDER BY DEPOSITDATE;"),
 dict(question="12 月的账单有哪些？最后缴款日是什么时候？",
      description="SELECT BILL_NUM, TENANTNAME, POSITIONNAME, SUBJECTNAME, BILLYEAR, BILLMONTH, BILL_DATE, LASTPAYDATE FROM bibillrecvinfo WHERE STOREID = 'STORE_DEMO_001' AND BILLMONTH = 12;"),
 dict(question="最近 7 天停车场车流和收入如何？",
      description="SELECT date_key, parking_entries, avg_turnover_times, parking_revenue_amount FROM fact_parking_daily ORDER BY date_key DESC LIMIT 7;"),
 dict(question="各项目空铺面积汇总？",
      description="SELECT project_name, count(*) AS vacant_cnt, sum(vacant_area) AS vacant_area FROM vw_mall_ops_vacancy_snapshot GROUP BY project_name ORDER BY vacant_area DESC;"),
 dict(question="当前哪些合同快到期了？",
      description="SELECT project_name, contract_no, brand_name, shop_name, lease_end_date, days_to_expiry FROM vw_mall_ops_contract_expiry WHERE days_to_expiry BETWEEN 0 AND 90 ORDER BY lease_end_date;"),
]
for i, ex in enumerate(SQL_EXAMPLES, 1):
    print(f"E{i:02d} {ex['question']}")

## 5. 对照 mallcre 模式与种子数据验证（可消费性的机器裁决）

三级验证：**① 对象/列存在性**（深度括号扫描解析 CREATE TABLE，视图取别名∪引用列）；**② description 绑定引用**（`obj.col` 对逐条核对）；**③ 种子字面量**（SQL 里的编码/状态字面量必须真实出现在种子文件）。先用 starter pack 自身（11 术语+18 示例）校准验证器——它过了，生成物的失败才可归因于生成物本身。

In [ ]:
SQL_KW = {"where","on","group","order","limit","left","right","inner","outer","full","cross","join",
          "window","union","set","values","using","partition","by","and","or","not","as","asc","desc",
          "between","in","select","from","when","then","case","else","end","having"}

def parse_schema(paths):
    objs = {}
    for p in paths:
        text = open(p).read()
        for m in re.finditer(r'create\s+table\s+(?:if\s+not\s+exists\s+)?"?(\w+)"?\s*\(', text, re.I):
            name = m.group(1).lower(); i = m.end() - 1; depth = 0
            while i < len(text):
                if text[i] == "(": depth += 1
                elif text[i] == ")":
                    depth -= 1
                    if depth == 0: break
                i += 1
            body = text[m.end():i]
            cols = {c.lower() for c in re.findall(
                r'^\s*"?(\w+)"?\s+(?:bigint|varchar|char|text|numeric|decimal|integer|int|date|timestamp|timestamptz|boolean|double|precision|float|bytea|jsonb|uuid|smallint|real|time)\b',
                body, re.I | re.M)}
            objs.setdefault(name, set()).update(cols)
        for m in re.finditer(r'create\s+(?:or\s+replace\s+)?view\s+(\w+)\s+as\b', text, re.I):
            name = m.group(1).lower()
            end = text.find(";", m.end()); body = text[m.end(): end if end > 0 else len(text)]
            aliases = {a.lower() for a in re.findall(r'\bas\s+(\w+)', body, re.I)}
            refs = {c.lower() for _, c in re.findall(r'(\w+)\.(\w+)', body)}
            objs.setdefault(name, set()).update(aliases | refs)
    return objs

objects = parse_schema([PG_SCHEMA, DEMO_SCHEMA])
print(f"模式解析：{len(objects)} 个对象（mallcre_postgres.sql + postgres_demo_schema.sql）")

def check_binding(ref, objs):
    o, c = ref.rsplit(".", 1); o, c = o.lower(), c.lower()
    if o not in objs: return f"对象 {o} 不存在"
    if c not in objs[o]: return f"{o}.{c} 缺列"
    return None

def sql_validate(sql, objs):
    errs, alias = [], {}
    for m in re.finditer(r'(?:from|join)\s+"?(\w+)"?(?:\s+(?:as\s+)?(\w+))?', sql, re.I):
        obj = m.group(1).lower(); al = (m.group(2) or "").lower()
        if obj not in objs: errs.append(f"对象 {obj} 不在模式")
        if al and al not in SQL_KW: alias.setdefault(al, obj)
    for a, c in re.findall(r'(\w+)\.(\w+)', sql):
        a, c = a.lower(), c.lower()
        obj = alias.get(a) or (a if a in objs else None)
        if obj and c not in objs.get(obj, set()): errs.append(f"{obj}.{c} 缺列")
    return errs

seed_text = open(SEED).read()
def seed_literal_miss(sql):
    lits = re.findall(r"'([^']+)'", sql)
    return [l for l in lits if l and (re.match(r'^(LOC|CONT|MALL|STORE|BLDG)_', l) or l in ("空置","已出租","F1","F2")) and l not in seed_text]

def validate_terms(terms, label):
    """逐组校验 description 中的 obj.col 引用 + 规格断言"""
    fails = []
    for t in terms:
        for a, c in re.findall(r'([a-z_]\w*)\.(\w+)', t["description"]):
            err = check_binding(f"{a}.{c}", objects)
            if err: fails.append((t["word"], f"{a}.{c}", err))
    print(f"{label}: {len(terms)} 组，description 绑定引用失败 {len(fails)}")
    for f in fails[:6]: print("   ✗", f)
    return fails

def validate_sqls(examples, label):
    fails = []
    for ex in examples:
        errs = sql_validate(ex["description"], objects)
        miss = seed_literal_miss(ex["description"])
        if errs or miss: fails.append((ex["question"], errs, miss))
    print(f"{label}: {len(examples)} 条，失败 {len(fails)}")
    for f in fails[:6]: print("   ✗", f)
    return fails

st_term_fails = validate_terms(starter_terms, "[校准] starter 术语")
st_sql_fails = validate_sqls(json.load(open(f"{PACK}/sql_examples.json")), "[校准] starter SQL 示例")

In [ ]:
gen_term_fails = validate_terms(generated, "[裁决] 生成术语")
bind_fails = []
for g in generated:
    for bc in g["bindings"]:
        err = check_binding(bc, objects)
        if err: bind_fails.append((g["word"], bc, err))
print(f"[裁决] 生成术语 bindings 显式锚点：{sum(len(g['bindings']) for g in generated)} 条，失败 {len(bind_fails)}")
for f in bind_fails[:6]: print("   ✗", f)

gen_sql_fails = validate_sqls(SQL_EXAMPLES, "[裁决] 生成 SQL 示例")
assert not st_term_fails and not st_sql_fails, "验证器校准失败：starter pack 自身未过，先修验证器"
assert not gen_term_fails and not bind_fails and not gen_sql_fails, "生成物验证失败"

# D2 验收断言：A101 问句拉出铺位组，组含 POSITION_CODE 映射与空置口径
g_pos = next(g for g in generated if g["word"] == "铺位")
assert "A101" in g_pos["other_words"] and "POSITION_CODE" in g_pos["description"] and "空置" in g_pos["description"]
q101 = "A101 铺位为什么不能出租？"
pulled = [g["word"] for g in generated + starter_terms
          if any(v in q101 for v in [g["word"]] + list(g["other_words"] or []))]
assert "铺位" in pulled
print(f"\n全部验证通过 ✅ ｜ D2 验收断言通过：A101 问句拉出组 = {pulled}（含 POSITION_CODE 映射与空置口径）")

## 6. 生成前后召回对比（覆盖率提升的量化）

In [ ]:
after_rows, after_sum = recall_stats(starter_terms + generated)
print(f"{'类别':<8}{'基线':>6}{'生成后':>8}")
for cat in ("BI-P0", "ERP-对象"):
    b, a = base_sum[cat], after_sum[cat]
    print(f"{cat:<8}{b[0]}/{b[1]:>4}{a[0]}/{a[1]:>6}")
print("\nERP-对象逐问对比：")
for br, ar in zip(base_rows, after_rows):
    if br["cat"] != "ERP-对象": continue
    b_hit = br["substr"] or br["vec"]; a_hit = ar["substr"] or ar["vec"]
    print(f"  {br['q'][:24]:<26} 基线={'命中' if b_hit else 'MISS':<4} → 生成后={'命中' if a_hit else 'MISS'} {sorted(set(a_hit))}")

## 7. 落盘产物 + 指纹 manifest（D7 护栏：生成物带 sha256_16）

In [ ]:
def sha16(p): return hashlib.sha256(open(p, "rb").read()).hexdigest()[:16]

term_out = [{k: g[k] for k in ("word", "other_words", "description")} for g in generated]
sql_out = [{"question": e["question"], "description": e["description"]} for e in SQL_EXAMPLES]
tp, sp, mp_ = f"{OUT_DIR}/term-aliases.generated.json", f"{OUT_DIR}/sql-examples.generated.json", f"{OUT_DIR}/alias-merge-suggestions.json"
json.dump(term_out, open(tp, "w"), ensure_ascii=False, indent=2)
json.dump(sql_out, open(sp, "w"), ensure_ascii=False, indent=2)
json.dump(merge_suggestions, open(mp_, "w"), ensure_ascii=False, indent=2)

manifest = dict(
    generated_at=datetime.datetime.now().isoformat(timespec="seconds"),
    producer="semantic-model v0.1.1 → LnkChatBI 术语库/SQL 示例生成器（W15-D3 实验3）",
    source=dict(ontology=dict(path=ONT_PATH, sha256_16=sha_ont),
                semantic_model=SM_YAML, version=SM["meta"]["version"]),
    scope=dict(specific_ds=True,
               datasource_ids="SET_AT_IMPORT（= mallcre / CRE BI Demo 数据源 id）",
               reason="术语库是 oid 级共享池，Semantic Model 是项目级资产，不圈作用域=往共享池倒项目私货（W15-D2 决策②）"),
    r2_guard="scenario_layer_frozen=true；生成源仅 aliases+terms；scenario 名仅用于排除校验（断言已过）",
    counts=dict(term_groups=len(term_out), aliases=sum(len(g["other_words"]) for g in term_out),
                sql_examples=len(sql_out), merge_suggestions=len(merge_suggestions),
                ontology_unique_terms=len(unique_terms), consumed_unique_terms=len(consumed)),
    validation=dict(term_binding_failures=len(gen_term_fails), explicit_anchor_failures=len(bind_fails),
                    sql_failures=len(gen_sql_fails),
                    validator_calibration=dict(starter_term_failures=len(st_term_fails),
                                               starter_sql_failures=len(st_sql_fails))),
    files={os.path.basename(p): sha16(p) for p in (tp, sp, mp_)},
)
mfp = f"{OUT_DIR}/import-manifest.json"
json.dump(manifest, open(mfp, "w"), ensure_ascii=False, indent=2)
print("落盘 4 个产物：")
for p in (tp, sp, mp_, mfp): print(f"  {p}  sha256_16={sha16(p)}")
print("\ncounts:", json.dumps(manifest["counts"], ensure_ascii=False))
print("validation:", json.dumps(manifest["validation"], ensure_ascii=False))

## 8. 可视化：消费漏斗 / 召回前后对比 / 验证通过率

In [ ]:
# 图1 消费漏斗（单位逐级不同，只看形状与断点）
labels = ["术语出现（复算口径）", "唯一术语词", "被组词归并（消费）", "生成组内别名", "生成组（父行）"]
vals = [total_occ, len(unique_terms), len(consumed),
        sum(len(g["other_words"]) for g in generated), len(term_out)]
fig, ax = plt.subplots(figsize=(9, 4.2))
ypos = np.arange(len(labels))[::-1]
ax.barh(ypos, vals, color=["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#CCB974"])
for yi, v in zip(ypos, vals):
    ax.text(v, yi, f" {v}", va="center", fontsize=10)
ax.set_yticks(ypos); ax.set_yticklabels(labels)
ax.set_xlim(0, max(vals) * 1.12)
ax.set_title("W15-D3 · 术语层 → LnkChatBI 可消费漏斗\n（mallcre demo 靶只覆盖部分域：消费率低是靶子问题，机制由 A101 组证明）")
fig.tight_layout(); fig.savefig(f"{NB_DIR}/w15d3_消费漏斗.png", dpi=150); plt.close(fig)

# 图2 生成前后召回命中率
cats = ["BI-P0", "ERP-对象"]
before = [base_sum[c][0] / base_sum[c][1] * 100 for c in cats]
after = [after_sum[c][0] / after_sum[c][1] * 100 for c in cats]
fig, ax = plt.subplots(figsize=(7, 4.2))
x = np.arange(len(cats)); w = 0.34
b1 = ax.bar(x - w/2, before, w, label="导入前基线（starter 11 组）", color="#4C72B0")
b2 = ax.bar(x + w/2, after, w, label="生成后（+14 组）", color="#55A868")
for bars in (b1, b2):
    for r in bars: ax.text(r.get_x() + r.get_width()/2, r.get_height() + 1.5, f"{r.get_height():.0f}%", ha="center", fontsize=10)
ax.set_xticks(x); ax.set_xticklabels(cats); ax.set_ylim(0, 118)
ax.set_ylabel("探针命中率 %"); ax.set_title("W15-D3 · 检索召回：导入前基线 vs 生成后（子串+向量双路）")
ax.legend(loc="lower right")
fig.tight_layout(); fig.savefig(f"{NB_DIR}/w15d3_召回前后对比.png", dpi=150); plt.close(fig)

# 图3 验证通过率（含验证器校准）
suites = ["starter 术语\n(校准)", "starter SQL\n(校准)", "生成术语", "生成 SQL"]
n_items = [len(starter_terms), 18, len(generated), len(SQL_EXAMPLES)]
n_fail = [len({f[0] for f in st_term_fails}), len(st_sql_fails), len(gen_term_fails) + len(bind_fails), len(gen_sql_fails)]
n_fail2 = [min(n, ni) for n, ni in zip(n_fail, n_items)]
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ok = [ni - nf for ni, nf in zip(n_items, n_fail2)]
ax.bar(suites, ok, label="通过", color="#55A868")
ax.bar(suites, n_fail2, bottom=ok, label="失败", color="#C44E52")
for i, (ni, nf) in enumerate(zip(n_items, n_fail2)):
    ax.text(i, ni + 0.3, f"{ni-nf}/{ni}", ha="center", fontsize=10)
ax.set_ylabel("条目数"); ax.set_ylim(0, max(n_items) * 1.2)
ax.set_title("W15-D3 · 对照 mallcre 模式/种子：三级验证通过率（全绿才算可消费）")
ax.legend()
fig.tight_layout(); fig.savefig(f"{NB_DIR}/w15d3_验证通过率.png", dpi=150); plt.close(fig)
print("图已保存：w15d3_消费漏斗.png / w15d3_召回前后对比.png / w15d3_验证通过率.png")

## 9. 实验结论：Today's Question 的另一半答案

**同一份 ontology，喂"术语库"和喂"表结构注释"差在哪？——差在通道能力，且本实验把差量化了：**

1. **归并能力**：表结构注释（M-Schema 通道）是表/字段级静态文本，ontology 里的 844 个唯一词进不去；术语库通道把 N 个唯一词归并进 14 个语义组——**命中任一别名拉全组**，A101 这种检索救不了的编码映射（D2 量化：子串 miss、向量 0.33 不过阈值）只有这条通道能确定性预喂。
2. **口径注入能力**：description 是 Rule 构件的注入口（D1 结论 L3 接口）——「空置双口径」「押金两粒度」「状态两套值域」这些警示写进表注释没人看得到组装时也不被拉取；写进术语 description，命中即进 prompt。
3. **作用域能力**：术语库有 specific_ds/datasource_ids 作用域闸（项目级资产不污染 oid 级共享池）；表结构注释天然绑定单一数据源，看似安全，实则意味着**它无法承载跨 ERP/BI 双侧的词-物归并**——而那正是 Semantic Model 的本职。
4. **代价**：术语库是易失资产（库内行，无版本无指纹）——所以本实验的 manifest 带 sha256_16 指纹锚 + 源头 ontology 指纹，**消费不改变 SoT 地位，只改变消费物的可追溯性**（分层 SoT 宪章的第一次实战）。

**诚实边界**：mallcre demo 靶只覆盖部分域，术语层消费率是个位数百分比——消费率低是靶子（demo schema）问题不是资产问题；但 A101 组证明机制闭环。场景层 15/102 冻结未消费（R2），Rule 构件只走了 description 注入这一条窄缝（L3），Policy 语义化留给 D4。

### 附：验证器 v0 范围声明（诚实边界）

- 只校验**限定名引用**（`别名.列` / `对象.列`）与 FROM/JOIN 对象存在性；裸列名未校验（需 select-item 解析，v0 未做）。
- 列名比对大小写不敏感（近似）——真实导入需确认 mallcre PG 侧带引号大写列的折叠行为（convert 脚本产物）。
- 视图列集 = select 别名 ∪ 引用列（超集近似）。
- 种子字面量校验仅覆盖编码/状态类字面量（LOC_/CONT_/MALL_/STORE_/BLDG_ 前缀与 空置/已出租/F1/F2）。